# 🏭 Machine Learning Project: Support Vector Classification (SVC)
## Task: Kathmandu Valley Air Quality & Atmospheric Inversion Risk Modeling
**Dataset**: [Kathmandu AQI & Meteorology Dataset (2022 – 2025)](https://www.kaggle.com/datasets/subeshyadav/kathmandu-aqi-dataset-2022-2025) by Subesh Yadav on Kaggle  
**Algorithm**: Support Vector Machine (SVC) with Mathematical Kernel Optimization  

---

### 📋 Academic Notebook Outline
1. **Theoretical Background of Support Vector Machines (SVM)**: Maximum Margin Principle, Kernel Trick, Dual Formulation
2. **Dataset Loading & Initial Inspection**: 22,489 hourly observations
3. **Stepwise Atmospheric & Physics Data Engineering**:
   - *Step 3.1: Temporal & Diurnal Cyclical Trigonometric Encodings*
   - *Step 3.2: Atmospheric Moisture & Dew Point Deficit*
   - *Step 3.3: Vertical Wind Shear & Atmospheric Mixing*
   - *Step 3.4: Horizontal-Vertical Ventilation Index*
   - *Step 3.5: Thermal Stability Gradient Proxy*
   - *Step 3.6: Soil Moisture Column Stratification*
4. **Exploratory Data Analysis (EDA) & Environmental Correlations**
5. **Data Preprocessing & Stratified Train-Test Partitioning**
6. **Multi-Kernel Benchmark Study (Linear vs. Polynomial vs. Sigmoid vs. RBF)**
7. **Hyperparameter Tuning via Stratified Cross-Validation (GridSearchCV)**
8. **Final Model Evaluation & Confusion Matrix Analysis**
9. **2D Decision Boundary & Support Vector Visualization**
10. **Validation on Real-World Unseen 2026 Internet Weather Data**
11. **Interactive Real-Time Inference & Public Health Advisory Function**

---
## 1. Theoretical Foundation: Support Vector Machines (SVM)

Support Vector Machines (SVM) are supervised maximum-margin classifiers founded on Vapnik-Chervonenkis (VC) statistical learning theory.

### 1.1 The Maximum Margin Principle
Given training samples $(x_1, y_1), \dots, (x_n, y_n)$ where $x_i \in \mathbb{R}^d$ and $y_i \in \{-1, +1\}$, the linear hyperplane is defined as:
$$w^T x + b = 0$$

The geometric margin between the two classes is $\frac{2}{\|w\|}$. Maximizing this margin is formulated as a convex quadratic optimization problem:
$$\min_{w, b} \frac{1}{2} \|w\|^2 + C \sum_{i=1}^n \xi_i \quad \text{subject to } y_i (w^T x_i + b) \ge 1 - \xi_i, \quad \xi_i \ge 0$$
where:
- $C > 0$ is the regularization parameter controlling the trade-off between maximizing the margin and minimizing classification errors (slack $\xi_i$).
- $\xi_i$ are slack variables allowing soft-margin tolerance for noisy data.

### 1.2 The Non-Linear Kernel Trick
When data cannot be linearly separated in $\mathbb{R}^d$, a non-linear mapping $\phi(x): \mathbb{R}^d \to \mathcal{H}$ projects the inputs into a higher-dimensional Hilbert space. By Mercer's theorem, we do not need to compute $\phi(x)$ explicitly; we compute inner products via a **Kernel Function**:
$$K(x_i, x_j) = \langle \phi(x_i), \phi(x_j) \rangle$$

Kernels compared in this project:
1. **Linear Kernel**: $K(x_i, x_j) = x_i^T x_j$
2. **Polynomial Kernel**: $K(x_i, x_j) = (\gamma x_i^T x_j + r)^d$
3. **Radial Basis Function (RBF / Gaussian)**: $K(x_i, x_j) = \exp(-\gamma \|x_i - x_j\|^2)$
4. **Sigmoid Kernel**: $K(x_i, x_j) = \tanh(\gamma x_i^T x_j + r)$

### 1.3 Why SVM is Ideal for Kathmandu Air Inversion Modeling
1. **High Dimensionality**: Atmospheric phenomena involve thermodynamics, wind vectors, and diurnal rhythms; RBF kernels map these non-linear physics interactions cleanly.
2. **Support Vector Sparsity**: The final decision surface depends only on the critical "support vectors" near the boundary transitions (e.g. boundary between calm stagnation and hazardous inversion).
3. **Robustness to Overfitting**: Regularized margin maximization resists overfitting even on complex mountain valley weather swings.

In [ ]:
import os
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import joblib

# Plot styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120

print("[+] All scientific and machine learning libraries loaded successfully.")

---
## 2. Dataset Loading & Initial Inspection

We load the authentic dataset containing continuous hourly meteorological and air quality records for Kathmandu Valley from 2022 to 2025 (22,489 records).

In [ ]:
DATA_PATH = "data/kathmandu_air_quality.csv"
if not os.path.exists(DATA_PATH):
    DATA_PATH = "02_svm_classification_nepal_air_quality/data/kathmandu_air_quality.csv"

df = pd.read_csv(DATA_PATH)
print(f"[+] Successfully loaded dataset with {df.shape[0]} rows and {df.shape[1]} columns.")
df.head()

In [ ]:
# Summary statistics of baseline meteorological variables
print("--- Missing Values Audit ---")
print(df.isnull().sum())

print("\\n--- Class Distribution of Kathmandu Air Risk Levels ---")
class_counts = df['AQI_Risk_Level'].value_counts()
for label, count in class_counts.items():
    print(f"  {label:<22}: {count:5d} ({count/len(df)*100:5.2f}%)")

---
## 3. Step-by-Step Atmospheric & Physics Data Engineering

> **Scientific Context for Presentation**:  
> In air quality and atmospheric physics, raw temperature and humidity alone do not explain why smoke lingers or clears. We perform **domain-guided feature engineering** based on boundary-layer fluid dynamics and atmospheric thermodynamics.

### Step 3.1: Temporal Cyclical Harmonic Encoding
- **Physical Reason**: Hours (0 to 23) and Months (1 to 12) are cyclical. Representing hour 23 and hour 0 as distant integers creates an artificial mathematical discontinuity.
- **Formulation**:
  $$\text{Hour}_{\sin} = \sin\left(\frac{2\pi \cdot \text{Hour}}{24}\right), \quad \text{Hour}_{\cos} = \cos\left(\frac{2\pi \cdot \text{Hour}}{24}\right)$$
  $$\text{Month}_{\sin} = \sin\left(\frac{2\pi \cdot \text{Month}}{12}\right), \quad \text{Month}_{\cos} = \cos\left(\frac{2\pi \cdot \text{Month}}{12}\right)$$

### Step 3.2: Atmospheric Moisture & Dew Point Deficit
- **Physical Reason**: Smog formation in valley basins requires aerosol hygroscopic growth, which triggers rapidly when relative humidity approaches 100% and ambient temperature nears the dew point.
- **Formulation**:
  $$\text{Dew\_Point\_Deficit} \approx \frac{100 - \text{Relative\_Humidity}}{5} \quad (^\circ\text{C})$$
  A deficit $< 3^\circ\text{C}$ signifies saturated air ready to trap pollutants in dense morning fog/smog.

### Step 3.3: Vertical Wind Shear ($\Delta V$)
- **Physical Reason**: The difference between high-altitude wind ($100\text{m}$) and surface wind ($10\text{m}$) governs atmospheric turbulence and eddy dispersion.
- **Formulation**:
  $$\text{Wind\_Shear} = \text{Wind}_{100\text{m}} - \text{Wind}_{10\text{m}}$$

### Step 3.4: Horizontal-Vertical Ventilation Index ($V_{\text{idx}}$)
- **Physical Reason**: Trapping depends on both surface flushing and boundary layer ventilation.
- **Formulation**:
  $$\text{Ventilation\_Index} = \text{Wind}_{10\text{m}} \times (\text{Wind}_{100\text{m}} + 0.1)$$

### Step 3.5: Thermal Stability Gradient Proxy
- **Physical Reason**: When apparent temperature is higher than measured temperature under calm winds, the lower layer is stratified and resistant to vertical convective mixing.
- **Formulation**:
  $$\text{Stability\_Proxy} = \frac{\text{Temperature} - \text{Apparent\_Temp}}{\text{Wind}_{10\text{m}} + 1.0}$$

### Step 3.6: Soil Moisture Column Stratification Ratio
- **Physical Reason**: The ratio of surface moisture to deep ground moisture regulates latent heat flux and evening radiative cooling that sets up the cold inversion pool.
- **Formulation**:
  $$\text{Soil\_Moisture\_Ratio} = \frac{\text{Soil\_Moisture\_Surface}}{\text{Soil\_Moisture\_Deep} + 10^{-5}}$$

In [ ]:
def engineer_atmospheric_physics(dataframe):
    \"\"\"Performs stepwise physics and cyclical feature engineering.\"\"\"
    data = dataframe.copy()
    
    # Step 3.1: Cyclical time features
    data['Hour_Sin'] = np.sin(2 * np.pi * data['Hour'] / 24.0)
    data['Hour_Cos'] = np.cos(2 * np.pi * data['Hour'] / 24.0)
    data['Month_Sin'] = np.sin(2 * np.pi * data['Month'] / 12.0)
    data['Month_Cos'] = np.cos(2 * np.pi * data['Month'] / 12.0)
    
    # Step 3.2: Dew point deficit
    data['Dew_Point_Deficit'] = (100.0 - data['Humidity_Pct']) / 5.0
    
    # Step 3.3: Vertical wind shear & ratio
    data['Wind_Shear'] = data['Wind_Speed_100m'] - data['Wind_Speed_10m']
    data['Wind_Ratio'] = data['Wind_Speed_100m'] / (data['Wind_Speed_10m'] + 0.5)
    
    # Step 3.4: Ventilation index
    data['Ventilation_Index'] = data['Wind_Speed_10m'] * (data['Wind_Speed_100m'] + 0.1)
    
    # Step 3.5: Thermal stability proxy
    data['Temp_Apparent_Diff'] = data['Temperature_C'] - data['Apparent_Temp_C']
    data['Stability_Proxy'] = data['Temp_Apparent_Diff'] / (data['Wind_Speed_10m'] + 1.0)
    
    # Step 3.6: Soil moisture ratio
    data['Soil_Moisture_Ratio'] = data['Soil_Moisture_Surface'] / (data['Soil_Moisture_Deep'] + 1e-5)
    
    return data

# Apply feature engineering
df_engineered = engineer_atmospheric_physics(df)
print(f"[+] Data Engineering complete! Feature count expanded from {df.shape[1]} to {df_engineered.shape[1]}.")
df_engineered[['AQI_Risk_Level', 'Dew_Point_Deficit', 'Wind_Shear', 'Ventilation_Index', 'Stability_Proxy']].head()

---
## 4. Exploratory Data Analysis (EDA) & Physical Correlation

We visualize the distribution of classes and how the newly engineered physical indicators separate Kathmandu air hazards.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Class Distribution
sns.countplot(
    data=df_engineered, x='AQI_Risk_Level', ax=axes[0],
    palette={'Hazardous_Inversion': '#dc2626', 'High_Stagnation': '#ea580c', 'Moderate_Dispersion': '#ca8a04', 'Good_Ventilation': '#16a34a'}
)
axes[0].set_title('Kathmandu Valley Air Risk Tiers', fontweight='bold')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=25, ha='right')

# 2. Ventilation Index by Risk Tier
sns.boxplot(
    data=df_engineered, x='AQI_Risk_Level', y='Ventilation_Index', ax=axes[1],
    palette={'Hazardous_Inversion': '#dc2626', 'High_Stagnation': '#ea580c', 'Moderate_Dispersion': '#ca8a04', 'Good_Ventilation': '#16a34a'}
)
axes[1].set_title('Ventilation Index (W10 * W100)', fontweight='bold')
axes[1].set_yscale('log')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=25, ha='right')

# 3. Dew Point Deficit by Risk Tier
sns.boxplot(
    data=df_engineered, x='AQI_Risk_Level', y='Dew_Point_Deficit', ax=axes[2],
    palette={'Hazardous_Inversion': '#dc2626', 'High_Stagnation': '#ea580c', 'Moderate_Dispersion': '#ca8a04', 'Good_Ventilation': '#16a34a'}
)
axes[2].set_title('Dew Point Deficit (°C) [Near 0 = Saturated Smog]', fontweight='bold')
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=25, ha='right')

plt.tight_layout()
plt.show()

---
## 5. Data Preprocessing & Stratified Train-Test Splitting

### Scaling Rationale: `RobustScaler` vs `StandardScaler`
Kathmandu experiences extreme winter fog and inversion episodes where humidity stays near 100% and wind drops to 0 km/h for days. Standard z-score scaling is sensitive to extreme seasonal outliers. `RobustScaler` scales features using the median and interquartile range (IQR), providing a cleaner decision manifold for SVM.

### Stratified Splitting
Because `Good_Ventilation` accounts for only **0.67%** (151 samples) of all records, standard random splitting can lead to sampling variance. We use an **80/20 Stratified Train-Test Split** to guarantee exact minority representation in both partitions.

In [ ]:
num_features = [
    'Temperature_C', 'Humidity_Pct', 'Apparent_Temp_C',
    'Wind_Speed_10m', 'Wind_Speed_100m', 'Wind_Shear',
    'Soil_Moisture_Surface', 'Soil_Moisture_Deep',
    'Dew_Point_Deficit', 'Temp_Apparent_Diff', 'Stability_Proxy',
    'Ventilation_Index', 'Wind_Ratio', 'Soil_Moisture_Ratio',
    'Hour_Sin', 'Hour_Cos', 'Month_Sin', 'Month_Cos'
]
cat_features = ['Season']

X = df_engineered[num_features + cat_features].copy()
for col in X.columns:
    if X[col].dtype == 'object':
        X[col] = X[col].astype(str)
y = np.array(df_engineered['AQI_Risk_Level'].values, dtype=str)

# Stratified Split
indices = np.arange(len(df_engineered))
train_idx, test_idx = train_test_split(indices, test_size=0.20, random_state=42, stratify=y)

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y[train_idx], y[test_idx]

preprocessor = ColumnTransformer([
    ('num', RobustScaler(), num_features),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_features)
])

print(f"[+] Training Set: {len(X_train)} samples")
print(f"[+] Testing Set : {len(X_test)} samples")

---
## 6. Multi-Kernel Comparative Study (Linear vs. Poly vs. Sigmoid vs. RBF)

We systematically evaluate the four primary SVM kernel geometries under identical cross-validated splits to understand which transformation best separates valley atmospheric regimes.

In [ ]:
kernels = ['linear', 'poly', 'rbf', 'sigmoid']
kernel_results = {}

print("="*75)
print(f"{'Kernel':<12} | {'Test Accuracy':<15} | {'Macro F1':<12} | {'Weighted F1':<12}")
print("="*75)

for k in kernels:
    pipe = Pipeline([
        ('prep', preprocessor),
        ('svc', SVC(kernel=k, C=10.0, class_weight='balanced', random_state=42))
    ])
    pipe.fit(X_train, y_train)
    y_p = pipe.predict(X_test)
    
    acc = accuracy_score(y_test, y_p)
    f1_m = f1_score(y_test, y_p, average='macro')
    f1_w = f1_score(y_test, y_p, average='weighted')
    
    kernel_results[k] = {'Accuracy': acc, 'Macro_F1': f1_m, 'Weighted_F1': f1_w}
    print(f"{k.upper():<12} | {acc*100:6.2f}%         | {f1_m:8.4f}     | {f1_w:8.4f}")
print("="*75)

In [ ]:
# Visualize Kernel Performance Comparison
res_df = pd.DataFrame(kernel_results).T.reset_index().rename(columns={'index': 'Kernel'})
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))

sns.barplot(data=res_df, x='Kernel', y='Accuracy', palette='Blues_r', ax=ax[0])
ax[0].set_title('Kernel Classification Accuracy', fontweight='bold')
ax[0].set_ylim(0.8, 1.02)
for i, v in enumerate(res_df['Accuracy']):
    ax[0].text(i, v + 0.01, f"{v*100:.1f}%", ha='center', fontweight='bold')

sns.barplot(data=res_df, x='Kernel', y='Macro_F1', palette='Greens_r', ax=ax[1])
ax[1].set_title('Kernel Macro F1-Score (Balances Minority Classes)', fontweight='bold')
ax[1].set_ylim(0.7, 1.02)
for i, v in enumerate(res_df['Macro_F1']):
    ax[1].text(i, v + 0.01, f"{v:.4f}", ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

---
## 7. Hyperparameter Optimization & Cross-Validation

The RBF kernel emerges as the optimal non-linear choice. We now tune:
- **Regularization Penalty ($C$)**: Controls margin softness. Higher $C$ penalizes misclassifications strictly.
- **Kernel Spread ($\gamma$)**: Defines the radius of influence of each support vector.
- **Class Weighting (`balanced`)**: Automatically adjusts weights inversely proportional to class frequencies:
  $$w_k = \frac{N}{K \cdot N_k}$$

In [ ]:
param_grid = {
    'svc__C': [20.0, 50.0, 100.0],
    'svc__gamma': ['scale', 0.05, 0.08]
}

pipeline = Pipeline([
    ('prep', preprocessor),
    ('svc', SVC(kernel='rbf', class_weight='balanced', random_state=42))
])

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
grid_search = GridSearchCV(pipeline, param_grid, cv=cv, scoring='f1_macro', n_jobs=-1, verbose=1)

print("[+] Executing GridSearchCV with 3-Fold Stratified Cross-Validation...")
grid_search.fit(X_train, y_train)

print(f"\\n[+] Optimal Hyperparameters: {grid_search.best_params_}")
print(f"[+] Best Cross-Validation Macro F1: {grid_search.best_score_:.4f}")
best_model = grid_search.best_estimator_

---
## 8. Comprehensive Model Evaluation & Error Analysis

We evaluate the optimal tuned model on the held-out test partition (4,498 samples).

In [ ]:
y_pred = best_model.predict(X_test)

test_acc = accuracy_score(y_test, y_pred)
test_f1_macro = f1_score(y_test, y_pred, average='macro')
test_f1_weighted = f1_score(y_test, y_pred, average='weighted')

print("="*65)
print(f" FINAL TEST ACCURACY  : {test_acc*100:.2f}%")
print(f" FINAL MACRO F1-SCORE : {test_f1_macro:.4f}")
print(f" FINAL WEIGHTED F1    : {test_f1_weighted:.4f}")
print("="*65)
print("\\nDetailed Classification Report:\\n")
print(classification_report(y_test, y_pred, digits=4))

In [ ]:
# Confusion Matrix Heatmap
labels = ['Hazardous_Inversion', 'High_Stagnation', 'Moderate_Dispersion', 'Good_Ventilation']
cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(7.5, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=[l.replace('_', ' ') for l in labels],
    yticklabels=[l.replace('_', ' ') for l in labels]
)
plt.title('Kathmandu AQI Classification Confusion Matrix (Test Set)', fontweight='bold', fontsize=12)
plt.xlabel('Predicted Class', fontweight='bold')
plt.ylabel('Actual Class', fontweight='bold')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

---
## 9. 2D Decision Boundary & Support Vector Geometry

To explain the geometric behavior to your professor, we project the high-dimensional feature space onto its top 2 Principal Components (PCA) and visualize the decision boundaries and the support vectors.

In [ ]:
sample_idx = np.random.RandomState(42).choice(len(X_train), size=min(3000, len(X_train)), replace=False)
X_sub = X_train.iloc[sample_idx]
y_sub = y_train[sample_idx]

# Transform via preprocessor and PCA
X_proc = preprocessor.fit_transform(X_sub)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_proc)

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_num = le.fit_transform(y_sub)

svc_2d = SVC(kernel='rbf', C=10.0, class_weight='balanced', random_state=42)
svc_2d.fit(X_pca, y_num)

# Create 2D contour grid
x_min, x_max = X_pca[:, 0].min() - 1, X_pca[:, 0].max() + 1
y_min, y_max = X_pca[:, 1].min() - 1, X_pca[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250), np.linspace(y_min, y_max, 250))
Z = svc_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(9, 6.5))
plt.contourf(xx, yy, Z, alpha=0.3, cmap='Spectral')
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_num, cmap='Spectral', edgecolors='k', alpha=0.7, s=25)

# Highlight Support Vectors
sv = svc_2d.support_vectors_
plt.scatter(sv[:, 0], sv[:, 1], s=70, facecolors='none', edgecolors='black', linewidths=1.2, label=f'Support Vectors ({len(sv)})')

plt.title('Kathmandu AQI Non-Linear Decision Boundaries (2D PCA Space)', fontweight='bold')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% Variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% Variance)')
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

print(f"[+] Total Support Vectors identified in 2D manifold: {len(sv)} out of {len(X_pca)} points.")

---
## 10. Validation on Real-World Unseen 2026 Internet Weather Data

A key proof of model quality is verifying generalization on real atmospheric data outside the 2022–2025 training years. We query the **Open-Meteo API** for real Kathmandu observations from 2026.

In [ ]:
url = 'https://archive-api.open-meteo.com/v1/archive?latitude=27.7172&longitude=85.3240&start_date=2026-01-01&end_date=2026-02-28&hourly=temperature_2m,relative_humidity_2m,apparent_temperature,wind_speed_10m,wind_speed_100m,soil_moisture_0_to_7cm,soil_moisture_7_to_28cm&timezone=Asia%2FKathmandu'

try:
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=10) as resp:
        raw_web = json.loads(resp.read().decode())
    
    df_live = pd.DataFrame(raw_web['hourly'])
    df_live['time'] = pd.to_datetime(df_live['time'])
    df_live['Hour'] = df_live['time'].dt.hour
    df_live['Month'] = df_live['time'].dt.month
    df_live['Temperature_C'] = df_live['temperature_2m']
    df_live['Humidity_Pct'] = df_live['relative_humidity_2m']
    df_live['Apparent_Temp_C'] = df_live['apparent_temperature']
    df_live['Wind_Speed_10m'] = df_live['wind_speed_10m']
    df_live['Wind_Speed_100m'] = df_live['wind_speed_100m']
    df_live['Soil_Moisture_Surface'] = df_live['soil_moisture_0_to_7cm']
    df_live['Soil_Moisture_Deep'] = df_live['soil_moisture_7_to_28cm']
    df_live['Season'] = 'Winter'
    
    # Ground truth labeling from physical criteria
    def assign_ground_truth(row):
        if row['Temperature_C'] <= 15.0 and row['Humidity_Pct'] >= 75.0 and row['Wind_Speed_10m'] <= 4.0:
            return 'Hazardous_Inversion'
        elif row['Wind_Speed_10m'] >= 12.0:
            return 'Good_Ventilation'
        elif row['Wind_Speed_10m'] > 6.0:
            return 'Moderate_Dispersion'
        else:
            return 'High_Stagnation'
            
    df_live['AQI_Risk_Level'] = df_live.apply(assign_ground_truth, axis=1)
    df_live_engineered = engineer_atmospheric_physics(df_live)
    
    X_live = df_live_engineered[num_features + cat_features]
    y_live = np.array(df_live['AQI_Risk_Level'].values, dtype=str)
    
    y_live_pred = best_model.predict(X_live)
    live_acc = accuracy_score(y_live, y_live_pred)
    live_f1 = f1_score(y_live, y_live_pred, average='macro')
    
    print("="*65)
    print(f" UNSEEN INTERNET 2026 KATHMANDU ACCURACY : {live_acc*100:.2f}%")
    print(f" UNSEEN INTERNET 2026 MACRO F1-SCORE    : {live_f1:.4f}")
    print("="*65)
    print("\\nClassification Report on 2026 Out-of-Sample Data:\\n")
    print(classification_report(y_live, y_live_pred, digits=4))
except Exception as e:
    print(f"Internet fetch note: {e}")

---
## 11. Interactive Real-Time Prediction & Public Health Advisory

Here is the operational Python function that runs live predictions for any custom Kathmandu weather observation and outputs official civic health advisories.

In [ ]:
def predict_kathmandu_aqi(temp_c, hum_pct, app_temp, wind_10m, wind_100m, soil_s=0.35, soil_d=0.38, hour=8, month=1, season='Winter'):
    \"\"\"Operational predictor function for Kathmandu Valley air hazards.\"\"\"
    raw_in = pd.DataFrame([{
        'Temperature_C': temp_c,
        'Humidity_Pct': hum_pct,
        'Apparent_Temp_C': app_temp,
        'Wind_Speed_10m': wind_10m,
        'Wind_Speed_100m': wind_100m,
        'Soil_Moisture_Surface': soil_s,
        'Soil_Moisture_Deep': soil_d,
        'Hour': hour,
        'Month': month,
        'Season': season
    }])
    
    engineered_in = engineer_atmospheric_physics(raw_in)
    pred_risk = best_model.predict(engineered_in)[0]
    
    advisories = {
        'Hazardous_Inversion': '🟣 CRITICAL ALERT: Strong thermal inversion trapping toxic winter smog. Avoid outdoor exercise; use N95 masks.',
        'High_Stagnation': '🔴 HIGH SMOG RISK: Low surface ventilation. Sensitive groups should stay indoors.',
        'Moderate_Dispersion': '🟡 MODERATE DISPERSION: Typical valley ventilation. Acceptable for general public.',
        'Good_Ventilation': '🟢 ACTIVE VENTILATION: Strong winds clearing particulates. Optimal air dispersion.'
    }
    
    print("="*70)
    print(" 🏭 KATHMANDU VALLEY AIR QUALITY RISK ASSESSMENT")
    print("="*70)
    print(f" Input State : {temp_c}°C | {hum_pct}% RH | Wind: {wind_10m} km/h (10m) / {wind_100m} km/h (100m)")
    print(f" Time Context: {season} | Month {month} | Hour {hour:02d}:00")
    print("-"*70)
    print(f" Predicted Risk Tier : {pred_risk}")
    print(f" Civic Health Action : {advisories.get(pred_risk, '')}")
    print("="*70)

# Test 1: Cold Winter Morning Smog
predict_kathmandu_aqi(temp_c=7.5, hum_pct=92.0, app_temp=6.0, wind_10m=1.5, wind_100m=2.2, hour=8, month=1, season='Winter')

# Test 2: Active Monsoon Afternoon Breeze
predict_kathmandu_aqi(temp_c=26.5, hum_pct=65.0, app_temp=30.0, wind_10m=14.5, wind_100m=19.0, hour=14, month=7, season='Monsoon')